# Chapter 06b — Optimization in 1D

Training a machine learning model means **minimizing** something — usually an
error. In one dimension we can already see every essential idea:

1. Minima and maxima happen where the slope is zero: $f'(x) = 0$.
2. We can **find** them by scanning a grid, or
3. by **gradient descent**: repeatedly step *downhill* opposite the slope.
4. The **learning rate** controls the step size — too small is slow, too large
   blows up.
5. As a bonus we meet **numerical integration** (the inverse idea: adding up
   slopes/areas).

This is the direct ancestor of how neural networks learn.

## 1. Setup

We bring back the central-difference derivative from notebook 06a.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # reproducible

def deriv(f, x, h=1e-5):
    # central difference estimate of f'(x); vectorized over arrays
    return (f(x + h) - f(x - h)) / (2 * h)

## 2. Minima and maxima sit where $f'(x) = 0$

At the bottom of a valley (a **minimum**) or the top of a hill (a **maximum**)
the curve is momentarily flat — its tangent is horizontal, so $f'(x) = 0$. These
flat spots are called **critical points**.

Consider

$$f(x) = x^4 - 3x^2 + x.$$

Let's plot $f$ and $f'$ and see where the derivative crosses zero.

In [ ]:
def f(x):
    return x**4 - 3*x**2 + x

xs = np.linspace(-2, 2, 400)

plt.plot(xs, f(xs),        label="f(x)")
plt.plot(xs, deriv(f, xs), label="f'(x)")
plt.axhline(0, color="gray", linewidth=0.8)
plt.title("Critical points: where f'(x) crosses zero")
plt.xlabel("x"); plt.ylabel("value")
plt.legend(); plt.grid(True)
plt.show()

Where $f'$ crosses zero going **up** (negative to positive), $f$ has a local
**minimum**; crossing **down** gives a local **maximum**.

## 3. Finding the minimum by scanning a grid

The bluntest method: evaluate $f$ at many points and keep the smallest. NumPy's
`argmin` returns the *index* of the minimum value, which we use to look up the
winning $x$.

In [ ]:
xs = np.linspace(-2, 2, 100001)   # a fine grid
ys = f(xs)

i = np.argmin(ys)                 # index of the smallest y
print("grid minimum near x =", round(xs[i], 4))
print("minimum value f(x)  =", round(ys[i], 4))

Scanning works but is wasteful: in real ML, $x$ may have *millions* of
components and we cannot grid them. We need a method that *follows the slope*.

## 4. Gradient descent

The derivative points in the direction of **steepest increase**. To go *down*,
step in the **opposite** direction. Repeat:

$$x \leftarrow x - \eta\, f'(x),$$

where $\eta$ (eta) is the **learning rate** — how big a step to take. Start
somewhere, take many small downhill steps, and you slide into a minimum.

In [ ]:
def gradient_descent(f, x_start, lr, n_steps):
    # returns the list of x-values visited (the "path")
    x = x_start
    path = [x]
    for _ in range(n_steps):
        slope = deriv(f, x)      # which way is uphill?
        x = x - lr * slope       # step the opposite way
        path.append(x)
    return np.array(path)

path = gradient_descent(f, x_start=1.5, lr=0.1, n_steps=40)
print("started at x =", path[0])
print("ended   at x =", round(path[-1], 4))
print("f at end      =", round(f(path[-1]), 4))

## 5. Plotting the descent path on the curve

Let's watch the steps. Each dot is one update; they march downhill into the
nearest valley.

In [ ]:
xs = np.linspace(-2, 2, 400)
plt.plot(xs, f(xs), label="f(x)", color="black")

# the path of points the optimizer visited:
plt.plot(path, f(path), "o-", color="red", markersize=4,
         label="gradient descent path")
plt.scatter([path[0]],  [f(path[0])],  color="green", zorder=5, label="start")
plt.scatter([path[-1]], [f(path[-1])], color="blue",  zorder=5, label="end")

plt.title("Gradient descent slides into a minimum")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.legend(); plt.grid(True)
plt.show()

Notice the dots are **far apart** when the slope is steep and **bunch up** as the
curve flattens near the bottom — the step size $\eta f'(x)$ naturally shrinks
because $f'(x)\to 0$ there.

## 6. The effect of the learning rate

The learning rate $\eta$ is the single most important knob:

- **Too small** — tiny steps, painfully slow, may not arrive in time.
- **Just right** — steady, quick convergence.
- **Too large** — steps overshoot the valley and can *diverge* (fly off).

Let's run the same descent from the same start with three learning rates and
track $f(x)$ over the steps.

In [ ]:
plt.figure()
for lr in [0.01, 0.1, 0.3]:
    path = gradient_descent(f, x_start=1.5, lr=lr, n_steps=40)
    plt.plot(f(path), "o-", markersize=3, label=f"lr = {lr}")

plt.title("Learning rate controls how fast (or whether) we converge")
plt.xlabel("step"); plt.ylabel("f(x) at current point")
plt.legend(); plt.grid(True)
plt.show()

In [ ]:
# A learning rate that is too big can DIVERGE. Watch the values explode:
path = gradient_descent(f, x_start=1.5, lr=0.6, n_steps=15)
print("x values with lr=0.6 (too large):")
print(np.round(path, 3))

With the careful rates the error falls smoothly; with `lr=0.6` the iterate
overshoots and the numbers blow up. This balance — fast but stable — is exactly
what ML practitioners tune every day.

## 7. Bonus: numerical integration (the inverse idea)

Differentiation measures slope; **integration** adds things up — the area under
a curve. They are inverses (the *Fundamental Theorem of Calculus*). The simplest
numerical method is the **trapezoid rule**: approximate the area by lots of thin
trapezoids. NumPy has it built in as `np.trapz`.

Let's check $\displaystyle\int_0^{\pi} \sin(x)\,dx = 2$.

In [ ]:
xs = np.linspace(0, np.pi, 1000)
ys = np.sin(xs)

area = np.trapz(ys, xs)          # trapezoid-rule area under sin from 0 to pi
print("numerical integral:", area)
print("exact value       :", 2.0)

In [ ]:
# Visualize the area we just measured
xs = np.linspace(0, np.pi, 200)
plt.plot(xs, np.sin(xs), color="black", label="sin(x)")
plt.fill_between(xs, np.sin(xs), alpha=0.3, label="area = integral")
plt.title("Integration adds up area under the curve")
plt.xlabel("x"); plt.ylabel("sin(x)")
plt.legend(); plt.grid(True)
plt.show()

So derivatives tell us *which way to step* (optimization) and integrals tell us
*how much accumulates* (areas, probabilities, totals). Both are everywhere in
machine learning.

---
## ✍️ Exercise 1 (solution included)

Minimize the simple parabola $f(x) = (x - 3)^2$ with gradient descent, starting
at $x = 0$ with learning rate $0.1$ for 50 steps. The minimum is obviously at
$x = 3$ — does the optimizer find it?

**Solution:**

In [ ]:
f = lambda x: (x - 3)**2
path = gradient_descent(f, x_start=0.0, lr=0.1, n_steps=50)
print("ended at x =", round(path[-1], 4), "(target 3)")
print("f at end   =", round(f(path[-1]), 6))

## ✍️ Exercise 2 (solution included)

Use `np.trapz` to estimate $\displaystyle\int_0^1 x^2\,dx$, whose exact value is
$\tfrac{1}{3}$. Try it with 5 grid points and again with 1000, and see how the
accuracy improves.

**Solution:**

In [ ]:
g = lambda x: x**2
for n in [5, 1000]:
    xs = np.linspace(0, 1, n)
    print(f"n={n:5d}  trapz = {np.trapz(g(xs), xs):.6f}   (exact 0.333333)")

---
## 📝 Homework (no solutions provided)

1. Run gradient descent on $f(x) = x^4 - 3x^2 + x$ from two different starts,
   $x_0 = -1.5$ and $x_0 = 1.5$. Do they reach the **same** minimum? What does
   this tell you about *local* minima?
2. For the parabola $f(x)=(x-3)^2$, find by experiment the largest learning rate
   that still converges. What happens just above it?
3. Write `minimize(f, x0, lr, tol)` that **stops automatically** once
   $|f'(x)| < \text{tol}$ instead of running a fixed number of steps. Report how
   many steps it took.
4. Estimate $\displaystyle\int_{-3}^{3} e^{-x^2}\,dx$ with `np.trapz`. (The exact
   value is close to $\sqrt{\pi}\approx 1.7725$.) How many grid points do you
   need for 4-decimal accuracy?